# PhysAR-IDS -- NB7 (v6): Defence 2 -- the autoencoder gate on its own

**Reviewer points this answers.**
1. *"Layers 'compound'. Section VI-B claims the two layers work better together than separately, but there is no result for the gate on its own."* -> sections 5-7 give the gate alone, Layer 1 alone and both, on clean data, under a gradient white-box attack on the gate, and burst by burst.
2. *"False alarms. A 5% false alarm rate on grid traffic is a lot of alerts. Discuss the practical impact."* -> section 4 converts false-positive rates into alerts per PLC-hour and shows what a persistence rule (alert only after m consecutive flagged bursts) costs and buys.
3. NB6 left two things open: the gate's clean false-positive rate on the temporal test split is about 11%, not the 5% it was calibrated to, and it detects only ~8% of FDI. This notebook finds out why and tests fixes.

**The gate (as in NB6).** Five autoencoders (5-fold) trained on benign training rows only; threshold tau = 95th percentile of the pooled out-of-fold error; a row is flagged when the ensemble-mean error is above tau. The NB6 gates are reloaded (`ae_gate_v6_seed{s}.pt`), nothing is retrained for the baseline.

**Why the calibration may be off (hypothesis, tested here).** Rows that share a (PLC, timestamp) are identical copies (~229 rows per burst). A *random* 5-fold split therefore puts copies of every held-out row in the training folds, so the out-of-fold error is too optimistic and tau is too low. A *time-blocked* split (each fold holds out a contiguous stretch of time per PLC) removes the copies. Section 3 measures the leak directly and compares both calibrations.

**Variants compared (all five seeds, same architecture and training budget; each new variant is trained with time-blocked folds):**
| name | gate input | loss weights | folds |
|---|---|---|---|
| `all_rand` | all 93 features | uniform | random (NB6, reloaded) |
| `all_blk` | all 93 | uniform | time-blocked |
| `phys_blk` | the 33 physical features only | uniform | time-blocked |
| `cyber_blk` | the 60 attacker-controlled cyber features only | uniform | time-blocked |
| `all_cyb3_blk` | all 93 | cyber features x3 | time-blocked |
| `all_phys3_blk` | all 93 | physical features x3 | time-blocked |

`phys_blk` matters for the threat model: the attacker only moves cyber features, so a gate that reads only physical features cannot be moved by the attack at all (checked numerically in the gradient attack, section 6).

**What is measured.** Ranking quality (AUC, overall and per class), calibration (test false-positive rate vs the nominal 5%, percentile sweep), alert burden, a gradient white-box attack against the gate alone (0.3 and 1.0 sigma on the 60 cyber features, 50 sign steps, best iterate kept), and gate vs Layer 1 vs both on clean data. Burst-level numbers use one observation per (class, PLC, timestamp). **Caution:** the test split holds only a handful of timestamps per series, so burst-level rates rest on few independent observations; the number is printed and a Wilson interval is given.

**Needs:** `ae_gate_v6_seed{s}.pt` (NB6), `xgb_robust_v6_seed{s}.json` (NB5b), `xgb_defended_v6_seed{s}.json` (NB5), the data files. **Runtime:** about 1 h in total on the local GPU; results are saved per seed.

## 0. Setup

In [ ]:
import os, sys, time, json, warnings, hashlib
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn
import xgboost as xgb
from scipy import stats
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import KFold
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
LAYER1 = {'rob': 'xgb_robust_{v}_seed{s}.json', 'def': 'xgb_defended_{v}_seed{s}.json'}
N_FOLDS = 5; GATE_EPOCHS = 30; GATE_HIDDEN = 32; GATE_BOTTLENECK = 12; GATE_PCTL = 95     # as NB6
PCTLS = [90, 95, 97.5, 99, 99.5, 99.9]                  # nominal thresholds for the calibration sweep
# name: (gate input, (group, weight) or None, folds)
VARIANTS = {'all_rand': ('all', None, 'nb6'), 'all_blk': ('all', None, 'blocked'), 'phys_blk': ('phys', None, 'blocked'), 'cyber_blk': ('cyber', None, 'blocked'),
            'all_cyb3_blk': ('all', ('cyber', 3.0), 'blocked'), 'all_phys3_blk': ('all', ('phys', 3.0), 'blocked')}
GRAD_BUDGETS = {'0.3': 0.3, '1.0': 1.0}; GRAD_STEPS = 50; GRAD_CHUNK = 32768       # gradient white-box on the gate alone (cyber features only)
PERSIST = [1, 2, 3]                                       # alert only after m consecutive flagged bursts of the same PLC
GRID = np.logspace(-3, 0, 31)                             # false-positive-rate grid for the ROC curves

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__}')
print('Working directory:', os.getcwd())

## 1. Load v6 data and the burst structure

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
PLC_train = np.asarray(data['PLCID_train']); PLC_test = np.asarray(data['PLCID_test']); ts_train = np.asarray(data['timestamp_train']); ts_test = np.asarray(data['timestamp_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; PHYS_IDX = np.setdiff1d(np.arange(N_FEAT), CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']; atk_names = class_names[1:]
FEAT_LO, FEAT_HI = X_train.min(0), X_train.max(0)
ben_tr = y_train == 0; Xb = np.ascontiguousarray(X_train[ben_tr]); plc_b, ts_b = PLC_train[ben_tr], ts_train[ben_tr]
benign_te = y_test == 0; atk_te = ~benign_te
print(f'features {N_FEAT} (cyber {len(CYBER_IDX)}, physical {len(PHYS_IDX)}) | benign train rows {len(Xb):,} | test rows {len(X_test):,} (benign {benign_te.sum():,}, attack {atk_te.sum():,})')

# ---- bursts: rows that share (class, PLC, timestamp) are identical copies ----
bid, buniq = pd.factorize(pd.MultiIndex.from_arrays([y_test, PLC_test, ts_test])); nB = len(buniq)
b_cls = np.zeros(nB, int); b_plc = np.zeros(nB, int); b_ts = np.zeros(nB); b_n = np.bincount(bid, minlength=nB)
b_cls[bid] = y_test; b_plc[bid] = PLC_test; b_ts[bid] = ts_test
sid = pd.factorize(pd.MultiIndex.from_arrays([b_cls, b_plc]))[0]; order = np.lexsort((b_ts, sid))
def burst_flag(f_row): return np.bincount(bid, weights=f_row.astype(float), minlength=nB) / b_n > 0.5
def persist(fb, m):
    f = fb[order]; s = sid[order]; out = f.copy()
    for lag in range(1, m):
        sh = np.roll(f, lag); same = np.roll(s, lag) == s; same[:lag] = False; out &= sh & same
    res = np.zeros_like(fb); res[order] = out; return res
# burst period from the benign series (train + test timestamps)
_t = pd.DataFrame({'p': np.r_[plc_b, PLC_test[benign_te]], 't': np.r_[ts_b, ts_test[benign_te]]}).drop_duplicates().sort_values(['p', 't'])
_d = _t.groupby('p')['t'].diff().dropna(); BURST_PERIOD = float(_d.median()); BURSTS_PER_HOUR = 3600.0 / BURST_PERIOD
N_BEN_BURSTS = int((b_cls == 0).sum())
print(f'test bursts: {nB} in total, {N_BEN_BURSTS} benign (the independent benign observations behind every burst-level rate) | rows per burst: median {int(np.median(b_n))}')
print(f'median spacing between consecutive benign bursts of a PLC: {BURST_PERIOD:.2f} s -> {BURSTS_PER_HOUR:.0f} bursts per PLC-hour')
print('attack bursts per class:', {c: int((b_cls == i + 1).sum()) for i, c in enumerate(atk_names)})
def wilson(k, n, z=1.96):
    if n == 0: return (float('nan'), float('nan'))
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d; return (c - h, c + h)

## 2. Gate code (inputs / weights / folds are options) and the gradient attack

In [ ]:
class AE(nn.Module):
    def __init__(self, n_in, bottleneck=GATE_BOTTLENECK, hidden=GATE_HIDDEN):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_in, hidden), nn.GELU(), nn.Linear(hidden, bottleneck), nn.GELU())
        self.decoder = nn.Sequential(nn.Linear(bottleneck, hidden), nn.GELU(), nn.Linear(hidden, n_in))
    def forward(self, x): return self.decoder(self.encoder(x))

class GateS:
    """ensemble of autoencoders reading the features `idx`; error = weighted MSE with weights `w`; flagged when error > tau."""
    def __init__(self, models, tau, idx, w):
        self.models = models; self.tau = float(tau); self.idx_np = np.asarray(idx); self.idx = torch.as_tensor(idx, device=DEVICE, dtype=torch.long); self.w = torch.as_tensor(w, device=DEVICE, dtype=torch.float32)
    def err_t(self, xb):
        z = xb[:, self.idx]; return torch.stack([(((m(z) - z) ** 2) * self.w).sum(1) / self.w.sum() for m in self.models]).mean(0)
    @torch.no_grad()
    def score(self, X, bs=65536):
        X = np.ascontiguousarray(X, dtype=np.float32); out = np.empty(len(X), np.float64)
        for s in range(0, len(X), bs): out[s:s + bs] = self.err_t(torch.from_numpy(X[s:s + bs]).to(DEVICE)).double().cpu().numpy()
        return out

def spec_of(inputs, wspec):
    idx = {'all': np.arange(N_FEAT), 'phys': PHYS_IDX, 'cyber': CYBER_IDX}[inputs]; w = np.ones(len(idx), np.float32)
    if wspec is not None: grp, val = wspec; w[np.isin(idx, CYBER_IDX) if grp == 'cyber' else ~np.isin(idx, CYBER_IDX)] = val
    return idx, w

def blocked_folds():
    """contiguous time blocks per PLC (benign train rows): fold k holds out block k of every PLC."""
    blk = np.zeros(len(ts_b), int)
    for p in np.unique(plc_b):
        m = plc_b == p; u = np.unique(ts_b[m]); r = np.searchsorted(u, ts_b[m]); blk[m] = np.minimum((r * N_FOLDS) // len(u), N_FOLDS - 1)
    return [(np.flatnonzero(blk != k), np.flatnonzero(blk == k)) for k in range(N_FOLDS)]
def random_folds(seed): return list(KFold(N_FOLDS, shuffle=True, random_state=seed).split(Xb))

def train_gate(seed, folds, idx, w):
    oof = np.zeros(len(Xb)); models = []; wt = torch.as_tensor(w, device=DEVICE)
    for k, (tr, ho) in enumerate(folds):
        torch.manual_seed(seed * 10 + k); m = AE(len(idx)).to(DEVICE); opt = torch.optim.Adam(m.parameters(), lr=1e-3)
        xt = torch.from_numpy(np.ascontiguousarray(Xb[tr][:, idx])).float().to(DEVICE); g = torch.Generator().manual_seed(seed * 10 + k)
        for ep in range(GATE_EPOCHS):
            m.train(); perm = torch.randperm(len(xt), generator=g).to(DEVICE)
            for s in range(0, len(xt), 256):
                xb = xt[perm[s:s + 256]]; opt.zero_grad(); (((m(xb) - xb) ** 2 * wt).sum(1) / wt.sum()).mean().backward(); opt.step()
        m.eval(); models.append(m); oof[ho] = GateS([m], 0, idx, w).score(Xb[ho])
    return GateS(models, np.percentile(oof, GATE_PCTL), idx, w), oof

def get_gate(name, seed):
    inputs, wspec, cv = VARIANTS[name]; idx, w = spec_of(inputs, wspec)
    if cv == 'nb6':                                                   # reload the NB6 gate; out-of-fold errors recomputed with the identical fold split
        ck = torch.load(f'ae_gate_{PIPELINE_VERSION}_seed{seed}.pt', map_location=DEVICE); ms_ = []
        for st in ck['state']: m = AE(N_FEAT).to(DEVICE); m.load_state_dict(st); m.eval(); ms_.append(m)
        oof = np.zeros(len(Xb))
        for k, (tr, ho) in enumerate(random_folds(seed)): oof[ho] = GateS([ms_[k]], 0, idx, w).score(Xb[ho])
        g = GateS(ms_, ck['tau'], idx, w); assert abs(np.percentile(oof, GATE_PCTL) - g.tau) <= 1e-4 * g.tau, 'NB6 threshold not reproduced'; return g, oof, False
    path = f'ae_gate7_{name}_{PIPELINE_VERSION}_seed{seed}.pt'; fp = repr((name, inputs, wspec, cv, N_FOLDS, GATE_EPOCHS, GATE_HIDDEN, GATE_BOTTLENECK, GATE_PCTL, seed, Xb.shape, float(Xb[:5].sum())))
    if os.path.exists(path):
        ck = torch.load(path, map_location=DEVICE, weights_only=False)
        if ck.get('fp') == fp:
            ms_ = []
            for st in ck['state']: m = AE(len(idx)).to(DEVICE); m.load_state_dict(st); m.eval(); ms_.append(m)
            return GateS(ms_, ck['tau'], idx, w), ck['oof'], False
    g, oof = train_gate(seed, blocked_folds() if cv == 'blocked' else random_folds(seed), idx, w)
    torch.save({'fp': fp, 'state': [m.state_dict() for m in g.models], 'tau': g.tau, 'idx': idx, 'w': w, 'oof': oof.astype(np.float32), 'name': name}, path); return g, oof, True

def gate_pgd(g, X, B, steps=GRAD_STEPS):
    """white-box attack on the gate alone: sign-gradient descent on the reconstruction error over the cyber features, L-inf ball B intersected with the training range, best iterate kept."""
    cyc = torch.as_tensor(CYBER_IDX, device=DEVICE); lo0 = torch.as_tensor(FEAT_LO, device=DEVICE); hi0 = torch.as_tensor(FEAT_HI, device=DEVICE); best_all = np.empty(len(X), np.float64); alpha = B / 10.0
    for s in range(0, len(X), GRAD_CHUNK):
        x0 = torch.from_numpy(np.ascontiguousarray(X[s:s + GRAD_CHUNK])).to(DEVICE); lo = x0.clone(); hi = x0.clone()
        lo[:, cyc] = torch.maximum(x0[:, cyc] - B, torch.minimum(lo0[cyc], x0[:, cyc])); hi[:, cyc] = torch.minimum(x0[:, cyc] + B, torch.maximum(hi0[cyc], x0[:, cyc]))
        x = x0.clone(); best = g.err_t(x0).detach()
        for _ in range(steps):
            x = x.detach().requires_grad_(True); e = g.err_t(x); grad, = torch.autograd.grad(e.sum(), x)
            with torch.no_grad():
                xn = x.detach().clone(); xn[:, cyc] = xn[:, cyc] - alpha * grad[:, cyc].sign(); xn = torch.minimum(torch.maximum(xn, lo), hi); best = torch.minimum(best, g.err_t(xn)); x = xn
        best_all[s:s + GRAD_CHUNK] = best.double().cpu().numpy()
    return best_all
print('gate code ready')

## 3. Per-seed run: train / reload gates, calibration, ranking, alert burden, gradient attack

In [ ]:
def op_stats(f):
    """row-level and burst-level numbers for a row-level flag vector f (benign flagged = false alarm)."""
    fb = burst_flag(f); r = {'row': {'fpr': float(f[benign_te].mean()), 'det_all': float(f[atk_te].mean()), 'det_pc': np.array([f[y_test == c].mean() for c in range(1, 6)])}, 'm': {}}
    for m in PERSIST:
        a = persist(fb, m); r['m'][m] = {'fpr_burst': float(a[b_cls == 0].mean()), 'alert_ph': float(a[b_cls == 0].mean() * BURSTS_PER_HOUR), 'det_pc': np.array([a[b_cls == c].mean() for c in range(1, 6)])}
    return r

R = {}; t_all = time.time()
for seed in SEEDS:
    pf = f'gate7_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = repr((VARIANTS, LAYER1, N_FOLDS, GATE_EPOCHS, GATE_PCTL, PCTLS, GRAD_BUDGETS, GRAD_STEPS, PERSIST, seed, X_test.shape, float(X_test[:5].sum()), 'nb7-v1'))
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: R[seed] = saved['R']; print(f'seed {seed}: result file found -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}', flush=True); r = {'gates': {}, 'l1': {}}
    nb6 = joblib.load(f'attack3_seed{seed}_{PIPELINE_VERSION}.pkl')['R']['gate_clean']
    # ---- Layer 1 (clean) predictions ----
    preds = {}
    for mtag, pat in LAYER1.items():
        cpu_m = xgb.XGBClassifier(); cpu_m.load_model(pat.format(v=PIPELINE_VERSION, s=seed)); cpu_m.set_params(device='cpu', n_jobs=-1); preds[mtag] = cpu_m.predict(X_test); r['l1'][mtag] = op_stats(preds[mtag] != 0)
        print(f'  Layer 1 [{mtag}] alone: false alarms {r["l1"][mtag]["row"]["fpr"]:.3f} | detection {r["l1"][mtag]["row"]["det_all"]:.3f} | per class {np.round(r["l1"][mtag]["row"]["det_pc"], 3).tolist()}', flush=True)
    for name in VARIANTS:
        ts_ = time.time(); g, oof, trained = get_gate(name, seed); sc = g.score(X_test); out = {'tau': g.tau}
        # ranking quality
        out['auc'] = float(roc_auc_score(atk_te, sc)); out['auc_pc'] = np.array([roc_auc_score(y_test[(y_test == 0) | (y_test == c)] == c, sc[(y_test == 0) | (y_test == c)]) for c in range(1, 6)])
        roc = []
        for c in range(0, 6):
            msk = np.ones(len(y_test), bool) if c == 0 else ((y_test == 0) | (y_test == c)); lab = (y_test != 0) if c == 0 else (y_test == c)
            fpr_, tpr_, _ = roc_curve(lab[msk], sc[msk]); roc.append(np.interp(GRID, fpr_, tpr_))
        out['roc'] = np.array(roc)                                                                    # row 0 = all attacks, rows 1..5 = per class
        thr5 = np.percentile(sc[benign_te], 95); out['tpr_at_test_fpr5_pc'] = np.array([(sc[y_test == c] > thr5).mean() for c in range(1, 6)])   # oracle threshold: ranking only, ignores calibration
        # calibration sweep (threshold = nominal percentile of the OOF error)
        out['sweep'] = {}
        for p in PCTLS:
            tp = np.percentile(oof, p); f = sc > tp; fb = burst_flag(f)
            out['sweep'][p] = {'tau': float(tp), 'fpr_row': float(f[benign_te].mean()), 'fpr_burst': float(fb[b_cls == 0].mean()), 'tpr_pc': np.array([f[y_test == c].mean() for c in range(1, 6)]), 'tpr_all': float(f[atk_te].mean())}
        out['oof_q'] = np.percentile(oof, PCTLS); out['test_benign_q'] = np.percentile(sc[benign_te], PCTLS)
        # operating point (tau at GATE_PCTL): gate alone and gate + Layer 1
        f0 = sc > g.tau; out['gate'] = op_stats(f0)
        for mtag in LAYER1: out[f'union_{mtag}'] = op_stats(f0 | (preds[mtag] != 0))
        # leak diagnostic: share of held-out benign training rows that have an identical copy (same PLC + timestamp) in the training folds
        key = pd.factorize(pd.MultiIndex.from_arrays([plc_b, ts_b]))[0]
        if VARIANTS[name][2] in ('nb6', 'random') and name == 'all_rand': out['leak'] = float(np.mean([np.isin(key[ho], key[tr]).mean() for tr, ho in random_folds(seed)]))
        if name == 'all_blk': out['leak'] = float(np.mean([np.isin(key[ho], key[tr]).mean() for tr, ho in blocked_folds()]))
        # gradient white-box on the gate alone
        out['grad'] = {}
        for bt, B in GRAD_BUDGETS.items():
            best = gate_pgd(g, X_test[atk_te], B); fl = best > g.tau; ya = y_test[atk_te]; out['grad'][bt] = {'det_all': float(fl.mean()), 'det_pc': np.array([fl[ya == c].mean() for c in range(1, 6)]), 'mean_err_ratio': float(np.mean(best) / max(np.mean(sc[atk_te]), 1e-12))}
        if name == 'all_rand': out['nb6_fpr'] = float(nb6['fpr_test']); assert abs(out['gate']['row']['fpr'] - out['nb6_fpr']) < 1e-3, 'baseline FPR differs from NB6'
        r['gates'][name] = out
        print(f'  [{name:13s}] {"trained" if trained else "loaded "} {(time.time() - ts_) / 60:4.1f} min | AUC {out["auc"]:.3f} per class {np.round(out["auc_pc"], 3).tolist()} | at nominal {GATE_PCTL}%: FPR row {out["gate"]["row"]["fpr"]:.3f} burst {out["gate"]["m"][1]["fpr_burst"]:.3f}, detection {out["gate"]["row"]["det_all"]:.3f} per class {np.round(out["gate"]["row"]["det_pc"], 3).tolist()} | detection after 1.0 sigma gradient attack {out["grad"]["1.0"]["det_all"]:.3f}', flush=True)
    R[seed] = r; joblib.dump({'fp': cfg_fp, 'R': r}, pf); print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min', flush=True)
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## 4. Results

In [ ]:
def ptest(a, b):
    d = np.asarray(a) - np.asarray(b)
    if len(d) < 2: return float('nan')
    return 1.0 if d.std(ddof=1) == 0 and d.mean() == 0 else (0.0 if d.std(ddof=1) == 0 else float(stats.ttest_rel(a, b).pvalue))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rk, i in enumerate(o): run = max(run, (len(p) - rk) * p[i]); adj[i] = min(1.0, run)
    return adj
ms = lambda x: f'{np.mean(x):.3f} +/- {(np.std(x, ddof=1) if len(x) > 1 else 0):.3f}'
GV = lambda v, f: np.array([R[s]['gates'][v][f] for s in SEEDS])
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 60)
T = {}      # tables saved to csv
print('4a. Baseline reproduction: reloaded NB6 gate, clean test false-positive rate', {s: round(R[s]['gates']['all_rand']['gate']['row']['fpr'], 4) for s in SEEDS}, '(asserted equal to the NB6 value)')
lk = [R[s]['gates']['all_rand']['leak'] for s in SEEDS]; lb = [R[s]['gates']['all_blk']['leak'] for s in SEEDS]
print(f'4b. Leak behind the calibration: share of held-out benign rows whose identical copy (same PLC + timestamp) is in the training folds -- random folds {ms(lk)}, time-blocked folds {ms(lb)}')

print('\n4c. Ranking quality (AUC, benign vs attack; per class = benign vs that class)')
rows = []
for v in VARIANTS: rows.append({'gate': v, 'AUC all attacks': ms(GV(v, 'auc')), **{f'AUC {c}': ms(GV(v, 'auc_pc')[:, i]) for i, c in enumerate(atk_names)}, 'TPR at test FPR 5%: all': ms(GV(v, 'tpr_at_test_fpr5_pc').mean(1)), **{f'TPR@5% {c}': ms(GV(v, 'tpr_at_test_fpr5_pc')[:, i]) for i, c in enumerate(atk_names)}})
T['ranking'] = pd.DataFrame(rows); print(T['ranking'].to_string(index=False))

print(f'\n4d. Calibration: test false-positive rate (benign rows) when the threshold is the nominal percentile of the out-of-fold benign error (ideal = 100 - nominal)')
rows = []
for v in VARIANTS: rows.append({'gate': v, **{f'nominal {100 - p:g}%': ms([R[s]['gates'][v]['sweep'][p]['fpr_row'] for s in SEEDS]) for p in PCTLS}})
T['calibration_fpr'] = pd.DataFrame(rows); print(T['calibration_fpr'].to_string(index=False))
print('   detection of attack rows at the same thresholds')
rows = []
for v in VARIANTS: rows.append({'gate': v, **{f'nominal {100 - p:g}%': ms([R[s]['gates'][v]['sweep'][p]['tpr_all'] for s in SEEDS]) for p in PCTLS}})
T['calibration_tpr'] = pd.DataFrame(rows); print(T['calibration_tpr'].to_string(index=False))
print('   per-class detection at the operating point (threshold = 95th percentile of the OOF error)')
rows = []
for v in VARIANTS: rows.append({'gate': v, 'FPR': ms([R[s]['gates'][v]['gate']['row']['fpr'] for s in SEEDS]), **{c: ms([R[s]['gates'][v]['gate']['row']['det_pc'][i] for s in SEEDS]) for i, c in enumerate(atk_names)}})
T['operating_point'] = pd.DataFrame(rows); print(T['operating_point'].to_string(index=False))
q = [(np.mean([R[s]['gates']['all_rand']['oof_q'][i] for s in SEEDS]), np.mean([R[s]['gates']['all_rand']['test_benign_q'][i] for s in SEEDS]), np.mean([R[s]['gates']['all_blk']['oof_q'][i] for s in SEEDS]), np.mean([R[s]['gates']['all_blk']['test_benign_q'][i] for s in SEEDS])) for i in range(len(PCTLS))]
print('   error quantiles (mean over seeds): percentile | all_rand OOF -> test benign | all_blk OOF -> test benign'); [print(f'     p{PCTLS[i]:g}: {a:.5f} -> {b:.5f} | {c:.5f} -> {d:.5f}') for i, (a, b, c, d) in enumerate(q)]
fr, fb_ = np.array([R[s]['gates']['all_rand']['gate']['row']['fpr'] for s in SEEDS]), np.array([R[s]['gates']['all_blk']['gate']['row']['fpr'] for s in SEEDS])
print(f'   paired test, FPR at the 5% operating point: all_rand {ms(fr)} vs all_blk {ms(fb_)}, p = {ptest(fr, fb_):.4f}')
pp = [ptest([R[s]['gates']['all_rand']['gate']['row']['det_pc'][i] for s in SEEDS], [R[s]['gates']['all_blk']['gate']['row']['det_pc'][i] for s in SEEDS]) for i in range(5)]; ph = holm(pp)
print('   per-class detection all_rand vs all_blk (Holm over 5 classes):', {c: (round(float(np.mean([R[s]["gates"]["all_rand"]["gate"]["row"]["det_pc"][i] for s in SEEDS])), 3), round(float(np.mean([R[s]["gates"]["all_blk"]["gate"]["row"]["det_pc"][i] for s in SEEDS])), 3), round(float(ph[i]), 4)) for i, c in enumerate(atk_names)})

print(f'\n4e. Alert burden. Burst-level: one observation per (PLC, timestamp); {N_BEN_BURSTS} independent benign test bursts, one burst every {BURST_PERIOD:.1f} s per PLC = {BURSTS_PER_HOUR:.0f} per PLC-hour.')
k_ = int(round(np.mean([R[s]['gates']['all_rand']['gate']['m'][1]['fpr_burst'] for s in SEEDS]) * N_BEN_BURSTS)); lo_, hi_ = wilson(k_, N_BEN_BURSTS)
print(f'   baseline all_rand: burst-level false-alarm rate {np.mean([R[s]["gates"]["all_rand"]["gate"]["m"][1]["fpr_burst"] for s in SEEDS]):.3f} (about {k_}/{N_BEN_BURSTS} bursts; 95% Wilson interval {lo_:.3f}-{hi_:.3f}); the row-level rate weights bursts by their row count')
rows = []
for kind in ['gate', 'union_rob', 'union_def']:
    for v in VARIANTS:
        for m in PERSIST:
            d = [R[s]['gates'][v][kind]['m'][m] for s in SEEDS]
            rows.append({'alarm': ('gate alone' if kind == 'gate' else f'Layer 1 [{kind[6:]}] + gate'), 'gate': v, 'consecutive bursts m': m, 'false alarms (bursts)': ms([x['fpr_burst'] for x in d]), 'alerts per PLC-hour': f'{np.mean([x["alert_ph"] for x in d]):.1f}', **{c: ms([x['det_pc'][i] for x in d]) for i, c in enumerate(atk_names)}})
for mtag in LAYER1:
    for m in PERSIST:
        d = [R[s]['l1'][mtag]['m'][m] for s in SEEDS]; rows.append({'alarm': f'Layer 1 [{mtag}] alone', 'gate': '-', 'consecutive bursts m': m, 'false alarms (bursts)': ms([x['fpr_burst'] for x in d]), 'alerts per PLC-hour': f'{np.mean([x["alert_ph"] for x in d]):.1f}', **{c: ms([x['det_pc'][i] for x in d]) for i, c in enumerate(atk_names)}})
T['alert_burden'] = pd.DataFrame(rows); print(T['alert_burden'][T['alert_burden']['gate'].isin(['-', 'all_rand', 'all_blk'])].to_string(index=False))

print('\n4f. The gate on its own, Layer 1 on its own, and both (clean test rows; row-level)')
rows = []
def add(label, f_fpr, f_det, f_pc):
    rows.append({'system': label, 'false alarms (benign rows)': ms(f_fpr), 'detection of attack rows': ms(f_det), **{c: ms([x[i] for x in f_pc]) for i, c in enumerate(atk_names)}})
for mtag in LAYER1: add(f'Layer 1 [{mtag}] alone', [R[s]['l1'][mtag]['row']['fpr'] for s in SEEDS], [R[s]['l1'][mtag]['row']['det_all'] for s in SEEDS], [R[s]['l1'][mtag]['row']['det_pc'] for s in SEEDS])
for v in ['all_rand', 'all_blk']:
    add(f'gate alone ({v})', [R[s]['gates'][v]['gate']['row']['fpr'] for s in SEEDS], [R[s]['gates'][v]['gate']['row']['det_all'] for s in SEEDS], [R[s]['gates'][v]['gate']['row']['det_pc'] for s in SEEDS])
    for mtag in LAYER1: add(f'Layer 1 [{mtag}] + gate ({v})', [R[s]['gates'][v][f'union_{mtag}']['row']['fpr'] for s in SEEDS], [R[s]['gates'][v][f'union_{mtag}']['row']['det_all'] for s in SEEDS], [R[s]['gates'][v][f'union_{mtag}']['row']['det_pc'] for s in SEEDS])
T['gate_vs_layer1'] = pd.DataFrame(rows); print(T['gate_vs_layer1'].to_string(index=False))
for v in ['all_rand', 'all_blk']:
    pp = [ptest([R[s]['gates'][v]['union_rob']['row']['det_pc'][i] for s in SEEDS], [R[s]['l1']['rob']['row']['det_pc'][i] for s in SEEDS]) for i in range(5)]; ph = holm(pp)
    print(f'   detection added by the gate on top of Layer 1 [rob] ({v}): ', {c: (round(float(np.mean([R[s]["gates"][v]["union_rob"]["row"]["det_pc"][i] - R[s]["l1"]["rob"]["row"]["det_pc"][i] for s in SEEDS])), 3), round(float(ph[i]), 4)) for i, c in enumerate(atk_names)}, '(mean gain, Holm-adjusted p)')

print('\n4g. White-box gradient attack on the gate alone (cyber features only): share of attack rows still flagged')
rows = []
for v in VARIANTS:
    rows.append({'gate': v, 'clean': ms([R[s]['gates'][v]['gate']['row']['det_all'] for s in SEEDS]), **{f'{bt} sigma': ms([R[s]['gates'][v]['grad'][bt]['det_all'] for s in SEEDS]) for bt in GRAD_BUDGETS},
                 **{f'{c} @1.0': ms([R[s]['gates'][v]['grad']['1.0']['det_pc'][i] for s in SEEDS]) for i, c in enumerate(atk_names)}})
T['gate_gradient_attack'] = pd.DataFrame(rows); print(T['gate_gradient_attack'].to_string(index=False))
if 'phys_blk' in VARIANTS: print('   check: the physical-only gate cannot be moved by cyber perturbations -> detection after attack equals clean detection:', all(abs(R[s]['gates']['phys_blk']['grad'][bt]['det_all'] - R[s]['gates']['phys_blk']['gate']['row']['det_all']) < 1e-9 for s in SEEDS for bt in GRAD_BUDGETS))

## 5. Figure and save

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 10)); cols = plt.cm.tab10(np.arange(len(VARIANTS)))
a = ax[0, 0]
for i, v in enumerate(VARIANTS): roc = np.mean([R[s]['gates'][v]['roc'][0] for s in SEEDS], 0); a.plot(GRID, roc, label=f'{v} (AUC {np.mean(GV(v, "auc")):.3f})', color=cols[i])
a.set_xscale('log'); a.set_xlabel('false-positive rate (benign rows)'); a.set_ylabel('detection of attack rows'); a.set_title('ROC of the gate alone (mean over seeds)'); a.legend(fontsize=8); a.grid(alpha=.3, which='both')
a = ax[0, 1]; nom = np.array([100 - p for p in PCTLS]) / 100
for i, v in enumerate(VARIANTS):
    m_ = np.array([np.mean([R[s]['gates'][v]['sweep'][p]['fpr_row'] for s in SEEDS]) for p in PCTLS]); sd = np.array([np.std([R[s]['gates'][v]['sweep'][p]['fpr_row'] for s in SEEDS], ddof=1) for p in PCTLS]); a.errorbar(nom, m_, yerr=sd, marker='o', label=v, color=cols[i], capsize=2)
a.plot([nom.min(), nom.max()], [nom.min(), nom.max()], 'k--', lw=1, label='ideal'); a.set_xscale('log'); a.set_yscale('log'); a.set_xlabel('nominal false-positive rate (1 - percentile of OOF error)'); a.set_ylabel('measured test false-positive rate'); a.set_title('Calibration'); a.legend(fontsize=8); a.grid(alpha=.3, which='both')
a = ax[1, 0]; w_ = 0.8 / len(VARIANTS); xs = np.arange(5)
for i, v in enumerate(VARIANTS): a.bar(xs + i * w_, [np.mean([R[s]['gates'][v]['gate']['row']['det_pc'][k] for s in SEEDS]) for k in range(5)], w_, yerr=[np.std([R[s]['gates'][v]['gate']['row']['det_pc'][k] for s in SEEDS], ddof=1) for k in range(5)], capsize=1.5, label=v, color=cols[i])
a.set_xticks(xs + 0.4 - w_ / 2); a.set_xticklabels(atk_names); a.set_ylabel('detection at the operating point'); a.set_title('Per-class detection of the gate alone (clean)'); a.legend(fontsize=7); a.grid(alpha=.3, axis='y'); a.set_ylim(0, 1.05)
a = ax[1, 1]
for i, v in enumerate(VARIANTS): a.bar(xs + i * w_, [np.mean([R[s]['gates'][v]['grad']['1.0']['det_pc'][k] for s in SEEDS]) for k in range(5)], w_, yerr=[np.std([R[s]['gates'][v]['grad']['1.0']['det_pc'][k] for s in SEEDS], ddof=1) for k in range(5)], capsize=1.5, label=v, color=cols[i])
a.set_xticks(xs + 0.4 - w_ / 2); a.set_xticklabels(atk_names); a.set_ylabel('still flagged'); a.set_title('After the 1.0 sigma gradient white-box attack on the gate'); a.grid(alpha=.3, axis='y'); a.set_ylim(0, 1.05)
plt.tight_layout(); plt.savefig(f'gate7_defence2_{PIPELINE_VERSION}.png', dpi=150); plt.show()

joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'tables': T, 'per_seed': {s: R[s] for s in SEEDS}, 'config': {'VARIANTS': VARIANTS, 'PCTLS': PCTLS, 'GRAD_BUDGETS': GRAD_BUDGETS, 'GRAD_STEPS': GRAD_STEPS, 'PERSIST': PERSIST, 'BURST_PERIOD': BURST_PERIOD, 'N_BEN_BURSTS': N_BEN_BURSTS}}, f'gate7_results_{PIPELINE_VERSION}.pkl')
for k, t in T.items(): t.to_csv(f'gate7_{k}_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB7', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'xgboost': xgb.__version__, 'torch': torch.__version__, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb7_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'gate7_results_{PIPELINE_VERSION}.pkl', f'gate7_defence2_{PIPELINE_VERSION}.png', f'nb7_run_manifest_{PIPELINE_VERSION}.json'] + [f'gate7_{k}_{PIPELINE_VERSION}.csv' for k in T] + [f'ae_gate7_{n}_{PIPELINE_VERSION}_seed{s}.pt' for n in VARIANTS if VARIANTS[n][2] != 'nb6' for s in SEEDS]: print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)